# Marketplace plan scrape notebook

- This uses the `scrapers` module to pull data from various marketplaces using configured input scenarios (e.g., single male, 3 dependents, etc.)

- The configuration file is where to configure each run. It contains:
1. Output directory
2. The scenarios to use
3. Max concurrency of calls
4. How to handle/backoff failures

- All scrapes write to a `<run_id>.raw.jsonl`, which is a listed JSON file containing all of the returned data from the scrape. It can later compiled into a CSV or other document using the compile notebook

- Module defaults (endpoints, page size) live in each module's `settings.py` and can be set per state with `"overrides"` in the config or by `MARKETPLACE_<MODULE>_<FIELD>` environment variables.

- Flow: config → each state's ZIPs resolved to counties → scenarios per county (household × income, from `scenario`) → the state's scraper module (`federal`, `virginia` or `maryland`) → one raw line per response

## Configuration

- `STATES` lists the states to run, in order, and the scraper module each uses
- Each state is its own run with its own files, `<timestamp>_<suffix>.raw.jsonl`
- The suffix is the state code unless you set `"suffix"`
- Comment a state out to skip it
- Everything else (scenario grid, request pacing, output folder) is in `marketplaces.json`

In [ ]:
import asyncio
import logging
from dataclasses import asdict
from itertools import islice
from pathlib import Path

import pandas as pd

from scrapers import HttpClient, Location, RunWriter, ScenarioGenerator, config_snapshot, load_config, make_scraper, state_run_id, utc_now

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
log = logging.getLogger("run")

CONFIG_PATH = Path("marketplaces.json")
ZIP_COUNTIES_DIR = Path("data/reference/zip_counties")  # one <state>.csv per state: which counties each ZIP is in

STATES = {
    # "AK": {"module": "federal"},
    # "AL": {"module": "federal"},
    "AR": {"module": "federal", "suffix": "AR"},
    "AZ": {"module": "federal", "suffix": "AZ"},
    "DE": {"module": "federal", "suffix": "DE"},
    "FL": {"module": "federal", "suffix": "FL"},
    "HI": {"module": "federal", "suffix": "HI"},
    "IA": {"module": "federal", "suffix": "IA"},
    "IN": {"module": "federal", "suffix": "IN"},
    "KS": {"module": "federal", "suffix": "KS"},
    "LA": {"module": "federal", "suffix": "LA"},
    "MI": {"module": "federal", "suffix": "MI"},
    "MO": {"module": "federal", "suffix": "MO"},
    "MS": {"module": "federal", "suffix": "MS"},
    "MT": {"module": "federal", "suffix": "MT"},
    "NC": {"module": "federal", "suffix": "NC"},
    "ND": {"module": "federal", "suffix": "ND"},
    "NE": {"module": "federal", "suffix": "NE"},
    "NH": {"module": "federal", "suffix": "NH"},
    "OH": {"module": "federal", "suffix": "OH"},
    "OK": {"module": "federal", "suffix": "OK"},
    "OR": {"module": "federal", "suffix": "OR"},
    "SC": {"module": "federal", "suffix": "SC"},
    "SD": {"module": "federal", "suffix": "SD"},
    "TN": {"module": "federal", "suffix": "TN"},
    "TX": {"module": "federal", "suffix": "TX"},
    "UT": {"module": "federal", "suffix": "UT"},
    "WI": {"module": "federal", "suffix": "WI"},
    "WV": {"module": "federal", "suffix": "WV"},
    "WY": {"module": "federal", "suffix": "WY"},
    # "VA": {"module": "virginia", "suffix": "VA-test"},  # optional keys: "suffix", "overrides"
}

config = load_config(CONFIG_PATH, states=STATES)
generators = {state: ScenarioGenerator.from_config(state, config) for state in config.states}
print("scenario:", config.scenario.model_dump())
print("http:", config.http)

## ZIPs → counties

The UI workflow starts with searching by location, usually ZIP, but the pricing happens at the county-level (a county with a FIPS code). A single zipcode can belong to several counties, so, this is more aligned with pricing boundaries, TIGER boundaries, and more.

The lookup is the scraper's `resolve`
- for `federal`, it uses the federal marketplace's own ZIP lookup HTTP endpoint, making one request per ZIP
- `resolve_local` is the offline alternative
- The result is cached per state in `ZIP_COUNTIES_DIR` and
reused on later runs
- Delete a state's file to look it up again
- The optional ZIP-level map at the end of
`analysis.ipynb` reads the same files to give each ZIP the plans of its county.

In [ ]:
async def resolve_state(state: str) -> pd.DataFrame:
    """One row per ZIP-county pair for the state: from its saved file, else by looking up every ZIP."""
    path = ZIP_COUNTIES_DIR / f"{state}.csv"
    if path.exists():
        return pd.read_csv(path, dtype=str)
    async with HttpClient(config.http) as http:
        scraper = make_scraper(state, http, config)
        found = await asyncio.gather(*(scraper.resolve(z, config.coverage_year) for z in generators[state].zips()))
    table = pd.DataFrame([asdict(county) for counties in found for county in counties], dtype=str)
    table["source"], table["retrieved_at_utc"] = scraper.name, utc_now()
    path.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(path, index=False)
    return table


for state, generator in generators.items():
    zips = {z.zip_code for z in generator.zips()}
    table = await resolve_state(state)
    table = table[table["zip_code"].isin(zips)]
    counties = table.drop_duplicates("county_fips")  # the first ZIP found in a county stands for it in requests
    generator.counties = [Location(c.state, c.zip_code, c.county_fips, c.county_name) for c in counties.itertuples()]
    print(
        f"{state}: {len(zips)} ZIPs ({len(zips - set(table['zip_code']))} with no county) -> {len(counties)} counties"
        f" x {len(generator.households())} households x {len(generator.fpl_multipliers)} incomes = {generator.count()} scenarios"
    )
print("total:", sum(g.count() for g in generators.values()), "scenarios")

## Run functions

- `run_county` quotes every household for one county
- `run` does that for every county of one state, `http.max_concurrency` counties at a time
- Each state writes to its own run files

In [ ]:
async def run_county(scraper, county: Location, generator, run_id: str, stats: dict, limit=None) -> None:
    """Quote every scenario for one county. A failed scenario is logged and skipped."""
    for scenario in islice(generator.scenarios(county), limit):
        try:
            plans = await scraper.fetch_plans(scenario, run_id)
        except Exception as err:
            stats["scenarios_failed"] += 1
            log.error("%s: %s", scenario.label(), err)
            continue
        if not plans:
            log.warning("%s: no plans returned", scenario.label())
        stats["scenarios_ok"] += 1
        stats["plans"] += len(plans)
    stats["counties"] += 1


async def run(state: str, limit=None) -> dict:
    """One run for one state: its own run id, raw file and HTTP client.

    `limit` (for the smoke test) caps both the counties and the scenarios per county.
    """
    run_id = state_run_id(config, state)
    generator = generators[state]
    counties = iter(generator.counties[:limit])  # shared by the workers; each takes the next county
    stats = {"state": state, "run_id": run_id, "counties": 0, "scenarios_ok": 0, "scenarios_failed": 0, "plans": 0}

    async def worker(scraper):
        for county in counties:
            await run_county(scraper, county, generator, run_id, stats, limit)
            log.info("%s: %d of %d counties done", state, stats["counties"], len(generator.counties))

    with RunWriter(config.output_dir, run_id, config=config_snapshot(config, state)) as writer:
        async with HttpClient(config.http) as http:
            scraper = make_scraper(state, http, config, raw_sink=writer.write_raw)
            await asyncio.gather(*(worker(scraper) for _ in range(config.http.max_concurrency)))
    log.info("run %s: %d responses -> %s", run_id, writer.response_count, writer.raw_path)
    return stats

## Smoke test

- Quotes one scenario for the first county of the first state
- Confirms the module responds before spending a full run
- Writes a small run file of its own

In [ ]:
await run(next(iter(config.states)), limit=1)

## Full run

- Runs the states in `STATES` one after another, every scenario of each county
- The ZIPs → counties cell above prints the counts
- If the loop is interrupted, comment out the finished states in `STATES` and re-run to resume

In [ ]:
summaries = []
for state in config.states:
    summaries.append(await run(state))
    print(summaries[-1])